# Two-mode HAPRE: test the recovered modal subspace

This benchmark leaves notebooks 05/06 and their physical-projection → phase-retrieval loops intact.
It uses the same preprocessing as 05. Each selected energy is tested independently:

1. Reconstruct measured + and − separately with one complex mode each.
2. Reconstruct their intensity sum with two independent complex modes, identical supports, and no physical projection.
3. Fit one constant complex 2×2 unitary matrix on the support, using the coherent references **only after retrieval**.

The separate helicities are validation references, not ground truth. Their own reconstruction errors and repeatability matter.
An arbitrary pixel-dependent mixture is forbidden. No gain, translation, phase ramp or twin correction is fitted.
Poor registration or distinct coherent reconstruction ambiguities can therefore also spoil comparison.
All blind seeds are reported; selection for plots uses only the summed-data residual.
Two small principal angles alone do not prove a unitary match: inspect complex error, ranks and power fractions too.
A nearly rank-one charge-dominated reference makes the weak magnetic direction especially sensitive to noise.

The [PRL paper](https://doi.org/10.1103/PhysRevLett.134.016704) motivates testing modal basis ambiguity.
Its ptychographic scan overlap is additional information absent from this per-energy HAPRE control.
The main paper was read; its supplemental material could not be accessed. This notebook does not claim to reproduce its initialization procedure.

Seed selection uses the measured-intensity residual **after object support projection**. The final Fourier constraint alone would force a near-zero residual without proving convergence.


In [ ]:
%matplotlib inline
from pathlib import Path
import sys, json
import h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import set_workers
ROOT = Path.cwd()
if (ROOT / "maxiv_phase_test").is_dir(): ROOT = ROOT / "maxiv_phase_test"
sys.path.insert(0, str(ROOT.parent / "library"))
import CCI_core as cci
import fth_phase_workflow as wf
import phase_retrieval_geometry as geometry
import phase_retrieval_universal as pr
import phase_retrieval_universal as universal
from scipy.ndimage import map_coordinates, binary_dilation
from retrieval_progress import retrieval_progress

DATA_ROOT = Path("/media/riccardo/EXT_16TB1/DATA/MAXIV/MaxIV2409")
RAW_FOLDER, AVERAGE_FOLDER = DATA_ROOT / "raw", DATA_ROOT / "proc"
SUPPORT_FILE = ROOT / "processed/Logs/data_recon_ImId_1143_maxiv.hdf5"
OUTPUT_FILE = ROOT / "processed/Logs/data_phase_summed_hyperspectral_2modes_maxiv.hdf5"
# Positive scan, negative scan, averaged dark references, exposure seconds.
# All pairs must describe the same sample state and illumination.
PAIRS = [
    (1152, 1150, (1151, 1153), 0.065),
    (1155, 1157, (1156, 1158), 0.070),
    (1162, 1160, (1159, 1161), 0.065),
    (1167, 1170, (1169, 1171), 0.100),
    (1175, 1173, (1172, 1174), 0.350),
    (1178, 1180, (1177, 1181), 1.000),
    (1185, 1183, (1177, 1181), 1.000),
    (1191, 1189, (1187, 1190), 0.800),
    (1196, 1194, (1193, 1195), 0.600),
    (1204, 1206, (1205, 1203), 0.500),
    (1235, 1232, (1231, 1234), 0.250),
    (1241, 1239, (1237, 1238), 0.250),
]
USE_PHYSICAL_PROJECTION = True
OUTER_ITERATIONS = 30
INNER_ITERATIONS = 10
PROJECTION_RELAXATION = 0.3
REFERENCE_ENERGY_EV = None  # Median energy defines the common reciprocal grid.
ENERGY_TOLERANCE_EV = 0.25
DETECTOR_CENTER = (1137, 1126)
BINNING = 2
CROP = 248
SATURATION = 0.75e4
USE_SAVED_PIXEL_MASK = False
BEAMSTOP_RADIUS = 0  # Detector pixels before crop/binning; 0 adds no circular mask.
FRAME_NORMALIZATION = 1  # Cached images are frame averages; divide by exposure below.
MODES = [1, 1]  # Or [1, 1, 2]: positive, negative, optional nonmagnetic background.
ALGORITHMS = ["HAPRE", "ER"]
ITERATIONS = [100, 30]  # Warmup; increase if needed.
INITIALIZATION_SEED = 0
FFT_WORKERS = 1

# Controls apply only to this independent benchmark.
VALIDATION_INDICES = [5]  # Expand to range(len(PAIRS)) after the single-energy check.
VALIDATION_SEEDS = [0, 1]
VALIDATION_ITERATIONS = [750, 50]
OUTPUT_FILE = ROOT / "processed/Logs/data_modal_subspace_validation.hdf5"


Load all energies on the same reciprocal grid as notebook 05; run retrieval only for VALIDATION_INDICES.

In [ ]:
def scan_snapshot(scan_id):
    path = RAW_FOLDER / f"2409_maxiv_{scan_id:05d}.h5"
    with h5py.File(path, "r") as handle:
        entry = handle[f"entry{scan_id}"]
        snapshot = entry["measurement/pre_scan_snapshot"]
        def scalar(name):
            if name not in snapshot:
                return None
            values = np.asarray(snapshot[name][()]).reshape(-1)
            if values.size == 0:
                return None
            if np.issubdtype(values.dtype, np.number):
                if not np.allclose(values, values[0]):
                    raise ValueError(f"Scan {scan_id}: {name} changes during acquisition")
                return float(values[0])
            return str(values[0])
        return {key: scalar(key) for key in ("energy", "helicity", "magOOP")}

if len(PAIRS) < 2:
    raise ValueError("Use at least two energy pairs for spectral coupling")
metadata, energies = [], []
for pos_id, neg_id, dark_ids, exposure_s in PAIRS:
    if not np.isfinite(exposure_s) or exposure_s <= 0 or FRAME_NORMALIZATION <= 0 or not dark_ids:
        raise ValueError("Require positive exposure/frame scaling and dark references")
    pair_meta = [scan_snapshot(pos_id), scan_snapshot(neg_id)]
    pair_energy = [m["energy"] for m in pair_meta]
    if any(e is None or not np.isfinite(e) or e <= 0 for e in pair_energy):
        raise ValueError("Missing/invalid energy")
    if abs(pair_energy[0]-pair_energy[1]) > ENERGY_TOLERANCE_EV:
        raise ValueError("Pair energies differ")
    metadata.append(pair_meta); energies.append(np.mean(pair_energy))
fields_meta = [m["magOOP"] for pair in metadata for m in pair]
if any(f is None for f in fields_meta) or not np.allclose(fields_meta, fields_meta[0]):
    raise ValueError("Require matching magnetic-field metadata across all pairs")
energies = np.asarray(energies)
if np.ptp(energies) <= ENERGY_TOLERANCE_EV:
    raise ValueError("Select distinct energies")
reference_energy = float(np.median(energies) if REFERENCE_ENERGY_EV is None else REFERENCE_ENERGY_EV)
if not np.isfinite(reference_energy) or reference_energy <= 0:
    raise ValueError("Invalid reference energy")
print("Energies:", energies, "reference:", reference_energy)
print("Helicity metadata (+, - manifest order):", [[m["helicity"] for m in pair] for pair in metadata])

saved = wf.load_data_dict(SUPPORT_FILE)
support_source = (np.asarray(saved["supportmask"]) != 0).astype(np.uint8)
saved_mask = np.asarray(saved["mask_pixel"]) != 0
if support_source.ndim != 2 or saved_mask.shape != support_source.shape:
    raise ValueError("Support and mask must be matching 2D arrays")
def load_average(scan):
    with h5py.File(AVERAGE_FOLDER / f"scan_{scan:04d}_avg.h5", "r") as handle:
        image = np.asarray(handle["image"][()]).squeeze().astype(float)
    if image.shape != support_source.shape: raise ValueError("Input/support shape mismatch")
    return image

def register_energy(image, excluded, energy):
    scale = reference_energy / energy
    if np.isclose(scale, 1, rtol=0, atol=1e-12):
        return image, excluded
    grid = np.indices(image.shape, dtype=float)
    origin = np.asarray(image.shape) // 2
    coords = origin[:, None, None] + scale * (grid-origin[:, None, None])
    clean = np.where(excluded, 0., image)
    warped = map_coordinates(clean, coords, order=1, mode="constant", cval=0.) * scale**2
    unsafe = binary_dilation(excluded, structure=np.ones((3,3)))
    mask = map_coordinates(unsafe.astype(float), coords, order=1, mode="constant", cval=1.) > 0
    return warped, mask

prepared_pairs, mask_pairs = [], []
for pair_index, (pos_id, neg_id, dark_ids, exposure_s) in enumerate(PAIRS):
    images, masks = [], []
    dark = np.mean([load_average(scan) for scan in dark_ids], axis=0)
    for channel, scan in enumerate((pos_id, neg_id)):
        raw = load_average(scan)
        image = wf.center_image(raw-dark, DETECTOR_CENTER, cci)
        mask = wf.center_image((raw >= SATURATION).astype(np.uint8), DETECTOR_CENTER, cci) != 0
        mask |= ~np.isfinite(image)
        if USE_SAVED_PIXEL_MASK: mask |= saved_mask
        y, x = np.indices(image.shape)
        if BEAMSTOP_RADIUS > 0:
            mask |= np.hypot(y-image.shape[0]//2, x-image.shape[1]//2) < BEAMSTOP_RADIUS
        image = np.nan_to_num(image)/(FRAME_NORMALIZATION*exposure_s)
        image, mask = register_energy(image, mask, metadata[pair_index][channel]["energy"])
        images.append(image.astype(np.float32)); masks.append(mask)
    pair_images, pair_masks, support, _, input_geometry = geometry.prepare(
        np.stack(images), np.stack(masks), support_source,
        dict(crop=CROP, binning=BINNING, roi=None, Nmodes=1))
    prepared_pairs.append(pair_images)
    mask_pairs.append(pair_masks)
    print(f"Prepared energy {pair_index+1}/{len(PAIRS)}",flush=True)
prepared = np.stack(prepared_pairs)
prepared_masks = np.stack(mask_pairs)
del prepared_pairs, mask_pairs
summed_intensity = prepared.sum(axis=1)
mask_pixel = np.any(prepared_masks != 0, axis=1)
mask_pixel |= ~np.isfinite(summed_intensity) | (summed_intensity < 0)
del images, masks
material_thickness = universal.largest_support_component(support).astype(float)
fig, axes = plt.subplots(1, 3, figsize=(12,4))
for ax, img, title in zip(axes, [np.log1p(np.maximum(summed_intensity[0],0)), support, material_thickness],
                         ["First measured sum", "Support", "Assumed material aperture"]):
    ax.imshow(img); ax.set_title(title)
plt.show()


In [ ]:
from modal_alignment import align_modes

def reconstruct_control(target, excluded, count, seed):
    rng = np.random.default_rng(seed)
    objects = support[None] * np.exp(1j*rng.uniform(-np.pi, np.pi, (count, *support.shape)))
    start = np.fft.ifftshift(np.fft.ifft2(np.fft.fftshift(objects, axes=(-2,-1))), axes=(-2,-1))
    valid = ~excluded
    start *= np.sqrt(target[valid].sum()/np.sum(abs(start[:,valid])**2))
    recipe = pr.default_phase_retrieval_recipe()
    recipe.update(modes=[1]*count, algorithm_list=["HAPRE", "ER"],
        number_iterations=VALIDATION_ITERATIONS, helicity=["sum"]*2,
        beta_zero=[.5]*2, beta_mode=["arctan", "const"],
        alpha_zero=[0.]*2, alpha_mode=["const"]*2,
        RL_its=[0]*2, RL_freqs=[10**9]*2, TV_freqs=[10**9]*2,
        plot_every=[10**9]*2, average_img=[1]*2, Fourier_last=[True]*2,
        output=[True]*2, Startimage=[start[0] if count == 1 else start, "sum"], Startgamma=[None]*2,
        crop=0, binning=1, roi=None, hologram_intensity_cutoff_vmin=-1, return_format="dict")
    result = pr.phase_retrieval_algorithm({"sum": target}, mask_pixel=excluded,
        supportmask=support, phase_retrieval_recipe=recipe)
    f = np.asarray(result["full_coherence"]["sum"]).reshape(count, *support.shape)
    # Fourier_last enforces measured data by construction. Score after support
    # projection instead, so selection tests simultaneous constraint satisfaction.
    obj = objects_from_fourier(f) * support[None]
    supported = np.fft.ifftshift(np.fft.ifft2(np.fft.fftshift(obj, axes=(-2,-1))), axes=(-2,-1))
    prediction = np.sum(abs(supported)**2, axis=0)
    residual = np.linalg.norm((prediction-target)[valid])/np.linalg.norm(target[valid])
    return f, float(residual)

def objects_from_fourier(f):
    return np.fft.ifftshift(np.fft.fft2(np.fft.fftshift(f, axes=(-2,-1))), axes=(-2,-1))

validation_results = {}
with set_workers(FFT_WORKERS):
    for index in VALIDATION_INDICES:
        # Identical usable pixels for all three reconstructions. Negative dark-
        # subtracted individual pixels cannot supply coherent amplitudes.
        excluded = mask_pixel[index] | np.any(~np.isfinite(prepared[index]) | (prepared[index]<0), axis=0)
        references, reference_errors = [], []
        for channel in range(2):
            trials = [reconstruct_control(prepared[index,channel], excluded, 1, 1000+100*channel+s)
                      for s in VALIDATION_SEEDS]
            best = min(trials, key=lambda item: item[1])  # Selection uses data only.
            references.append(best[0][0]); reference_errors.append([t[1] for t in trials])
        reference = np.stack(references)
        ref_objects = objects_from_fourier(reference)
        trials = []
        for seed in VALIDATION_SEEDS:
            blind, residual = reconstruct_control(summed_intensity[index], excluded, 2, seed)
            obj = objects_from_fourier(blind)
            aligned, unitary, metrics = align_modes(obj, ref_objects, support != 0)
            # Material-only fit is a separate diagnostic, with its own constant U.
            _, _, material_metrics = align_modes(obj, ref_objects, material_thickness != 0)
            trials.append(dict(seed=seed, fields=blind, aligned_objects=aligned,
                               unitary=unitary, residual=residual, metrics=metrics,
                               material_metrics=material_metrics))
            print(index, seed, "support-projected sum residual", residual, "support metrics", metrics,
                  "material metrics", material_metrics, flush=True)
        validation_results[index] = dict(reference=reference, reference_errors=reference_errors,
                                         excluded=excluded, trials=trials)


In [ ]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
with h5py.File(OUTPUT_FILE, "w") as handle:
    handle.attrs["purpose"] = "Held-out unitary benchmark; no physical projection"
    handle.attrs["iterations_json"] = json.dumps(VALIDATION_ITERATIONS)
    handle.create_dataset("support", data=support)
    handle.create_dataset("material_thickness", data=material_thickness)
    for index, result in validation_results.items():
        group = handle.create_group(str(index))
        group.attrs["energy_ev"] = energies[index]
        group.attrs["reference_errors_json"] = json.dumps(result["reference_errors"])
        group.create_dataset("reference_fourier", data=result["reference"])
        group.create_dataset("excluded", data=result["excluded"])
        group.create_dataset("measured_helicities", data=prepared[index])
        for trial in result["trials"]:
            g = group.create_group(f"seed_{trial['seed']}")
            for name in ["fields", "aligned_objects", "unitary"]:
                g.create_dataset(name, data=trial[name])
            g.attrs["support_projected_sum_residual"] = trial["residual"]
            for name in ["metrics", "material_metrics"]:
                g.attrs[name+"_json"] = json.dumps(trial[name])
        best = min(result["trials"], key=lambda t:t["residual"])
        reference = objects_from_fourier(result["reference"])
        fig, axes = plt.subplots(2,4,figsize=(14,7))
        for c in range(2):
            for ax, image, title in zip(axes[c],
                [abs(reference[c]), abs(best["aligned_objects"][c]), np.angle(reference[c]), np.angle(best["aligned_objects"][c])],
                ["Reference amplitude", "Aligned blind amplitude", "Reference phase", "Aligned blind phase"]):
                ax.imshow(np.where(support, image, np.nan)); ax.set_title(title)
        fig.suptitle(f"Energy {energies[index]:.3f} eV; best seed by support-projected sum residual")
        plt.show()
print("Saved", OUTPUT_FILE)
